# Where can this project go?
### A strategy notebook: the crossover hypothesis, a working model prototype, and a plan

---

You have a paradigm that does not yet work, a hypothesis about implicit learning, and a model
that makes a prediction. This notebook is for deciding **what the project actually is**, before
spending months collecting data.

The central claim it develops is one you already wrote down without naming it:

> **Temporal coherence and spike-timing-dependent plasticity depend on onset asynchrony in
> opposite directions.** Coherence binds best when onsets coincide and fails as they separate.
> STDP *cannot operate* at zero lag — with simultaneous onsets there is no temporal order to
> learn — and strengthens as lag grows, up to the STDP window of roughly 20–40 ms.
>
> Therefore the benefit of a **consistent component order** must be **zero at step 0 and grow
> with asynchrony**. That crossover, if it exists, is the finding.

This is a much stronger claim than "how much asynchrony can a figure tolerate", because it is a
*mechanistic dissociation* with a quantitative prediction tied to a known biophysical time
constant, and it inverts the usual assumption that learning operates on already-segregated objects.

### What is in here

| § | | why it matters |
|---|---|---|
| 1 | The crossover, made quantitative and interactive | turns your idea into a falsifiable curve |
| 2 | Hear the contrast that carries the story | ordered vs reshuffled, at several asynchronies |
| 3 | A design flaw, demonstrated and fixed | your hypothesis needs learning; your design forbids it |
| 4 | **A working STDP model prototype** | does the model in your abstract actually do what you say? |
| 5 | Power: how many listeners, how many trials | decide feasibility before committing |
| 6 | What would falsify this, and the decision tree | know your exit conditions in advance |

*Nothing here is a result. It is a set of instruments for deciding what to measure.*

In [ ]:
#@title Setup (run me first)
import os, sys, subprocess

REPO_URL, REPO_DIR = "https://github.com/MeysamAmirsardari/SeqSFG_task.git", "SeqSFG_task"
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
ROOT = os.path.abspath(REPO_DIR) if os.path.isdir(REPO_DIR) else os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

from seqsfg import config, stimulus, measure
from seqsfg.stimulus import make_trial, render_interval, render_trial, FIGURE, BACKGROUND

cfg = config.DEFAULT
D   = config.validate(cfg)
SR  = cfg.sample_rate
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": False})

def play(x, label=""):
    if label: display(Markdown(f"**{label}**"))
    display(Audio(np.asarray(x, float), rate=SR, normalize=False))

print(f"repo: {ROOT}")
print(f"pool: {D.n_channels} channels, {D.channel_freqs_hz[0]:.0f}-{D.channel_freqs_hz[-1]:.0f} Hz")
print(f"max step at this rate: {int((cfg.iei_min_ms - cfg.tone_dur_ms)/(cfg.n_components-1))} ms "
      f"(elements must fit inside one inter-element interval)")

---

## 1. The crossover, made quantitative

Two mechanisms, two shapes. Write them down and the experiment designs itself.

**Coherence** binds components that start together. Its strength should fall as onsets separate,
on a timescale set by the integration window over which the auditory system treats onsets as
"simultaneous" — tens of milliseconds:

$$c(\Delta) = e^{-\Delta/\tau_c}$$

**Sequence learning by STDP** requires a lag. At $\Delta = 0$ there is no order to learn, so its
contribution is exactly zero. It grows as the lag enters the plasticity window and decays once
the lag exceeds it:

$$s(\Delta) = \frac{\Delta}{\tau_s} e^{\,1 - \Delta/\tau_s}$$

A listener hearing a figure with **reshuffled** order each repetition can use only coherence.
A listener hearing a **consistent** order can use both. So:

$$d'_{\text{shuffled}} = A\,c(\Delta), \qquad d'_{\text{ordered}} = A\,c(\Delta) + B\,s(\Delta)$$

**The order effect is the difference, and it is forced to be zero at $\Delta=0$.** That is not a
modelling convenience — it is a fact about your stimulus, which I verified in your code: at step 0
every component starts in the same instant, so "order" is undefined. You have a built-in zero point,
which is the cleanest possible control.

In [ ]:
#@title The predicted curves — move the sliders   { run: "auto" }
TAU_COHERENCE_MS = 12   #@param {type:"slider", min:4, max:40, step:1}
TAU_STDP_MS      = 20   #@param {type:"slider", min:5, max:50, step:1}
A_COHERENCE      = 2.5  #@param {type:"slider", min:0.5, max:4.0, step:0.1}
B_LEARNING       = 1.2  #@param {type:"slider", min:0.0, max:3.0, step:0.1}
MAX_STEP_MS      = 40   #@param {type:"slider", min:20, max:60, step:5}

d = np.linspace(0, MAX_STEP_MS, 400)
coh  = np.exp(-d / TAU_COHERENCE_MS)
stdp = (d / TAU_STDP_MS) * np.exp(1 - d / TAU_STDP_MS)
shuffled, ordered = A_COHERENCE * coh, A_COHERENCE * coh + B_LEARNING * stdp

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(d, coh, label=r"coherence  $e^{-\Delta/\tau_c}$", lw=2)
ax[0].plot(d, stdp, label=r"STDP  $(\Delta/\tau_s)e^{1-\Delta/\tau_s}$", lw=2)
ax[0].set(xlabel="onset asynchrony $\\Delta$ (ms)", ylabel="normalised strength",
          title="the two mechanisms\nopposite dependence on asynchrony")
ax[0].legend(fontsize=8); ax[0].axvline(0, color="0.8", lw=0.8)

ax[1].plot(d, ordered, lw=2.5, color="#1f77b4", label="consistent order (both mechanisms)")
ax[1].plot(d, shuffled, lw=2.5, color="#d62728", ls="--", label="reshuffled order (coherence only)")
ax[1].set(xlabel="onset asynchrony $\\Delta$ (ms)", ylabel="predicted d'",
          title="what listeners should do")
ax[1].legend(fontsize=8)

diff = ordered - shuffled
ax[2].plot(d, diff, lw=2.5, color="#2ca02c")
ax[2].fill_between(d, 0, diff, color="#2ca02c", alpha=0.2)
peak = d[np.argmax(diff)]
ax[2].axvline(peak, color="0.4", ls=":")
ax[2].annotate(f"peak at {peak:.0f} ms", xy=(peak, diff.max()), xytext=(peak+3, diff.max()*0.85), fontsize=9)
ax[2].set(xlabel="onset asynchrony $\\Delta$ (ms)", ylabel="d'(ordered) - d'(shuffled)",
          title="THE PREDICTION\nzero at 0 by construction, peaks in the STDP window")
plt.tight_layout(); plt.show()

print(f"Largest order benefit: d' = {diff.max():.2f} at {peak:.0f} ms asynchrony.")
print(f"At step 0 the benefit is exactly {diff[0]:.3f} - it CANNOT be otherwise, because with")
print("simultaneous onsets there is no order to preserve or reshuffle.")
print("\nThat zero point is your strongest control: any order effect you measure at step 0 is bias.")

### Why this shape is worth chasing

A monotonic decline (the original "how far can we shear it?" framing) is what *everyone* expects
and what a dozen mechanisms predict. It is a parameter measurement.

A **non-monotonic order benefit that is zero at synchrony and peaks near the STDP window** is a
signature. Very few accounts predict it, and it is quantitative: the peak location should track
the plasticity time constant, which is independently constrained by physiology. If your peak lands
at 3 ms or 300 ms, the STDP story is wrong and you will know.

Concretely, the analysis is an **order × asynchrony interaction**, with a predicted shape, and
a built-in zero. That is a strong test.

---

## 2. Hear the contrast that carries the story

Your code already builds this. `scrambled` keeps one delay order for every element of a trial;
`redrawn` reshuffles the order at every element. Same components, same channels, same asynchrony,
same everything — only the *consistency of order* differs.

At step 0 they are physically identical (nothing to order). Listen down the asynchronies and ask
yourself: at what point does the consistent one start to cohere into a thing, and the reshuffled
one stay a scatter?

**If you cannot hear a difference at any asynchrony, the project's central claim is in trouble,
and you have learned that in ninety seconds rather than nine months.**

In [ ]:
#@title Consistent order vs reshuffled order, at four asynchronies
for step in (0.0, 10.0, 20.0, 28.0):
    tr_ord = make_trial(cfg, seed=31337, step_ms=step, variant="scrambled", d=D)  # one order, every element
    tr_shf = make_trial(cfg, seed=31337, step_ms=step, variant="redrawn",   d=D)  # new order each element
    display(Markdown(f"### asynchrony = {step:g} ms" +
                     ("  — *physically identical: no order exists at zero*" if step == 0 else "")))
    play(render_interval(cfg, tr_ord.recurring, D), "consistent order")
    play(render_interval(cfg, tr_shf.recurring, D), "reshuffled every element")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), sharey=True)
for ax, variant, title in ((axes[0], "scrambled", "consistent order"),
                           (axes[1], "redrawn", "reshuffled every element")):
    tr = make_trial(cfg, 31337, 20.0, variant, d=D)
    t0 = tr.recurring.element_onsets[1] * cfg.grid_ms / 1000 - 0.05
    from seqsfg import plots as P
    P._raster(ax, cfg, D, tr.recurring, t0, t0 + 0.75, lw=3.5,
              recurring_lines=tr.recurring.figure_set)
    ax.set(title=f"{title} (20 ms asynchrony)", xlabel="time (s)")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("semitones re 1 kHz")
plt.tight_layout(); plt.show()
print("Left: the same rising staircase every time. Right: a different jumble every time.")
print("The components, channels and asynchrony are identical. Only the ORDER differs.")

---

## 3. A design flaw, demonstrated — and the fix

Your hypothesis is about **implicit learning**. Learning requires repeated exposure to *the same*
thing. Your stimulus draws a **new figure on new channels every single trial**.

So the only learning available is the six repetitions inside one trial. Nothing accumulates across
trials, because there is nothing in common to accumulate. If your abstract says "accuracy increased
over trials", that can only be general task familiarity — not learning of a figure.

Run the cell. Then look at the fix.

In [ ]:
#@title Is there anything to learn across trials?
sets = [tuple(make_trial(cfg, seed=2000+t, step_ms=10.0, variant="rising", d=D).recurring.figure_set.tolist())
        for t in range(12)]
print("figure channels, trial by trial (current design):")
for i, s in enumerate(sets[:6]): print(f"   trial {i+1}: {s}")
print(f"   ...\n   distinct sets in 12 trials: {len(set(sets))}/12")
overlap = np.mean([len(set(a) & set(b)) for a in sets for b in sets if a != b])
print(f"   mean channels shared by any two trials: {overlap:.1f} of {cfg.n_components} (chance level)")
print("\n=> Across trials there is NO recurring structure. Cross-trial learning is impossible.")
print("=> Exposures to any given pattern: 6 (within one trial), then it is gone forever.\n")

# ---- the fix: hold one pattern fixed across a block ----
def fixed_figure_trial(seed, step_ms, variant, anchor_seed=999):
    """A trial whose figure sits on a FIXED channel set, so it can be learned across trials."""
    anchor = np.random.default_rng(anchor_seed)
    S = stimulus.sample_figure_set(anchor, D.n_channels, cfg.n_components, cfg.figure_min_spacing_channels)
    for attempt in range(50):
        rng = np.random.default_rng([int(seed), attempt, 0xA5F6])
        try:
            A = stimulus.build_recurring(rng, cfg, D, step_ms, variant)
            # relabel A's figure onto the anchor set by swapping channel identities
            mapping = {int(old): int(new) for old, new in zip(A.figure_set, S)}
            inv = {v: k for k, v in mapping.items()}
            newchan = A.channel.copy()
            for j, c in enumerate(A.channel):
                c = int(c)
                if c in mapping:   newchan[j] = mapping[c]
                elif c in inv:     newchan[j] = inv[c]
            A.channel = newchan; A.figure_set = S.copy()
            A.element_sets = [S.copy() for _ in A.element_sets]
            other = (stimulus.build_ungrouped(rng, cfg, D, A) if variant in ("ungrouped", "onechannel")
                     else stimulus.build_redrawn(rng, cfg, D, A))
            return stimulus.Trial(seed=seed, variant=variant, step_ms=step_ms,
                                  recurring=A, other=other, n_rebuilds=attempt)
        except stimulus.PlacementError:
            continue
    raise RuntimeError("could not build")

fixed = [tuple(fixed_figure_trial(3000+t, 10.0, "rising").recurring.figure_set.tolist()) for t in range(6)]
print("figure channels with a FIXED anchor pattern (the fix):")
for i, s in enumerate(fixed[:4]): print(f"   trial {i+1}: {s}")
print(f"   distinct sets in 6 trials: {len(set(fixed))}")
n_block = 40
print(f"\n=> Now a listener gets {cfg.n_elements} x {n_block} = {cfg.n_elements*n_block} exposures to ONE pattern")
print("   across a 40-trial block, instead of 6. THAT is a learning experiment.")
print("\n   Note this is a prototype by channel-relabelling; the clean implementation is a")
print("   `figure_anchor_seed` config field so the whole battery can verify it.")

### What this unlocks

Once one pattern recurs across a block, three new experiments become available that you cannot
currently run at all:

1. **Learning curve.** Detection of the anchored figure should improve across the block while
   detection of novel figures does not. That is a within-session learning effect with its own
   control.
2. **Transfer.** After learning a pattern at one asynchrony, test it at another. If learned
   structure genuinely helps binding, the benefit should transfer across asynchrony — and that
   distinguishes "learned the object" from "learned a specific timing template".
3. **The strongest version of your claim.** Find the asynchrony at which a *novel* figure is at
   chance. Then ask whether a *learned* figure is detectable there. If yes, learning is not merely
   modulating segregation — it is **enabling** grouping that bottom-up cues cannot produce. That is
   the headline result, and it is a single well-chosen condition, not a sweep.

---

## 4. A working STDP model prototype

Your abstract claims a recurrent excitatory–inhibitory model whose lateral connections change by
rate-based STDP, and that the learned coincidence matrix becomes dominated by its leading
eigenvalue **in proportion to step size**. Let us find out whether that is true.

Before the model, one thing this exercise exposed about your *stimulus*, which matters more than
the model does.

### Your three variants are three learning timescales

| variant | order within a trial | order across trials | what can be learned |
|---|---|---|---|
| `rising` | identical every element | **identical every trial** (always ascending) | long-term sequence structure |
| `scrambled` | identical every element | **new permutation each trial** | within-trial structure only |
| `redrawn` | new every element | new every trial | nothing |

This was almost certainly not deliberate, and it is a gift: you have a three-level manipulation of
*how far back* structure must be retained, not merely a condition and its control.

**But it carries a confound you must fix.** `rising` is always ascending in frequency, so
"consistent order" is perfectly confounded with "sweeps upward in pitch". Rising and falling sweeps
are not perceptually equivalent, so counterbalance ascending and descending patterns across
listeners or trials before running anything.

In [ ]:
#@title The model
def channel_activity(c, d, iv, bin_ms=5.0):
    """Binary channel-by-time activation of one interval, straight from the schedule."""
    n_bins = int(c.interval_dur_ms / bin_ms)
    r = np.zeros((d.n_channels, n_bins))
    dur = max(1, int(round(c.tone_dur_ms / bin_ms)))
    for onset, ch in zip(iv.onset, iv.channel):
        b = int(onset * c.grid_ms / bin_ms)
        r[ch, b:min(b + dur, n_bins)] = 1.0
    return r

def stdp_delta(r, tau_bins=4.0, a_plus=1.0, a_minus=0.55):
    """Rate-based STDP: W[i,j] grows when j is active shortly BEFORE i, shrinks when after.
    At zero lag the causal and anti-causal terms cancel, so no order can be learned."""
    lags = np.arange(1, int(5 * tau_bins) + 1)
    kern = np.exp(-lags / tau_bins)
    trace = np.zeros_like(r)
    for k, w in zip(lags, kern):
        trace[:, k:] += w * r[:, :-k]
    dW = a_plus * (r @ trace.T) - a_minus * (trace @ r.T)
    np.fill_diagonal(dW, 0.0)
    return dW / max(r.shape[1], 1)

def train(c, d, step_ms, variant, n_trials=30, eta=0.01, tau_bins=4.0, seed=2):
    """eta is deliberately small: with a large one the weights saturate at the clip
    ceiling and the asymmetry we are trying to measure is masked."""
    W = np.zeros((d.n_channels, d.n_channels))
    for t in range(n_trials):
        tr = make_trial(c, seed * 10007 + t, step_ms, variant, d=d)
        W += eta * stdp_delta(channel_activity(c, d, tr.recurring), tau_bins)
    return W

def upward_bias(W):
    """Net 'earlier -> later' structure. 'rising' always ascends in frequency, so learned
    forward connections show up as lower-triangle (to-high, from-low) minus upper."""
    n = W.shape[0]
    il, iu = np.tril_indices(n, -1), np.triu_indices(n, 1)
    return float(W[il].mean() - W[iu].mean())

def eigen_domination(c, d, W, step_ms, variant, n_probe=10, gain=1.4, seed=99):
    doms = []
    for t in range(n_probe):
        r = channel_activity(c, d, make_trial(c, seed * 7919 + t, step_ms, variant, d=d).recurring)
        rr = r + gain * (W @ r)
        rr = rr - rr.mean(axis=1, keepdims=True)
        lam = np.clip(np.linalg.eigvalsh(np.cov(rr)), 0, None)
        doms.append(lam.max() / max(lam.sum(), 1e-12))
    return float(np.mean(doms))

print("model ready. NOTE: measure structure along the TEMPORAL order, not the channel index -")
print("in 'scrambled' the permutation differs per trial, so a channel-index measure reads as noise.")

### Does ordered exposure build directional structure?

The mechanism check. After exposure to figures whose components always arrive in the same order,
does the model develop connections that run **forward** along that order and not backward?

At step 0 it must not, because there is no order — the causal and anti-causal terms cancel exactly.
That is the model's own built-in control, and it matches the stimulus's built-in zero point.

In [ ]:
#@title Directional learning, across asynchrony (~1 min)
STEPS = [0, 5, 10, 15, 20, 28]
bias = {v: [] for v in ("rising", "scrambled", "redrawn")}
for s in STEPS:
    for v in bias:
        bias[v].append(upward_bias(train(cfg, D, float(s), v)))
    print(f"  step {s:2d} ms done", flush=True)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for v, c_, lbl in (("rising", "#1f77b4", "rising (order fixed across trials)"),
                   ("scrambled", "#ff7f0e", "scrambled (fixed within a trial only)"),
                   ("redrawn", "#d62728", "redrawn (never consistent)")):
    ax[0].plot(STEPS, bias[v], "o-", lw=2, color=c_, label=lbl)
ax[0].axhline(0, color="0.6", lw=0.8)
ax[0].set(xlabel="onset asynchrony (ms)", ylabel="forward - backward weight",
          title="directional structure learned\n(must be ~0 at step 0)")
ax[0].legend(fontsize=7.5)

d_ord = np.array(bias["rising"]) - np.array(bias["redrawn"])
ax[1].plot(STEPS, d_ord, "o-", lw=2.5, color="#2ca02c")
ax[1].fill_between(STEPS, 0, d_ord, color="#2ca02c", alpha=0.2)
ax[1].axhline(0, color="0.6", lw=0.8)
pk = STEPS[int(np.argmax(d_ord))]
ax[1].axvline(pk, color="0.4", ls=":")
ax[1].set(xlabel="onset asynchrony (ms)", ylabel="rising - redrawn",
          title=f"THE MODEL'S PREDICTION\npeak at {pk} ms")
plt.tight_layout(); plt.show()

print(f"\nzero point   : {d_ord[0]:+.6f}  at step 0 (must be ~0 - it is the model's own control)")
print(f"peak benefit : {d_ord.max():+.6f} at {pk} ms")
print(f"at 28 ms     : {d_ord[-1]:+.6f}")
print(f"""
READ THIS CAREFULLY. The phenomenological sketch in section 1 assumed the benefit peaks at the
STDP time constant (~20 ms). This mechanistic model does NOT agree: it peaks at {pk} ms and then
DECLINES. The reason is that a rate-based rule with an exponentially decaying causal kernel
potentiates most at the shortest non-zero lag, once the zero-lag cancellation is escaped.

That disagreement is the most useful thing in this notebook. The PEAK LOCATION is a sharp,
discriminating test:
  - benefit peaks near 5 ms  -> consistent with this rate-STDP implementation
  - benefit peaks near 20 ms -> consistent with a classical STDP window, NOT with this model
  - benefit is flat          -> neither; the effect is familiarity, not sequence order
Derive the prediction from YOUR model before collecting data, and preregister the peak.""")

### The eigenvalue claim from your abstract

Now the specific thing you wrote: leading-eigenvalue domination increasing with step size.

In [ ]:
#@title Eigenvalue domination across asynchrony (~2 min)
dom_o, dom_s = [], []
for s in STEPS:
    Wo, Ws = train(cfg, D, float(s), "rising"), train(cfg, D, float(s), "redrawn")
    dom_o.append(eigen_domination(cfg, D, Wo, float(s), "rising"))
    dom_s.append(eigen_domination(cfg, D, Ws, float(s), "redrawn"))
    print(f"  step {s:2d} ms done", flush=True)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(STEPS, dom_o, "o-", lw=2, color="#1f77b4", label="rising (consistent order)")
ax[0].plot(STEPS, dom_s, "s--", lw=2, color="#d62728", label="redrawn (no order)")
ax[0].set(xlabel="onset asynchrony (ms)", ylabel=r"$\lambda_1/\sum\lambda$",
          title="unified-representation index")
ax[0].legend(fontsize=8)
diff = np.array(dom_o) - np.array(dom_s)
ax[1].plot(STEPS, diff, "o-", lw=2.5, color="#2ca02c")
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].fill_between(STEPS, 0, diff, color="#2ca02c", alpha=0.2)
ax[1].set(xlabel="onset asynchrony (ms)", ylabel="ordered - shuffled",
          title="the order benefit in the eigenvalue readout")
plt.tight_layout(); plt.show()

slope = np.polyfit(STEPS, dom_o, 1)[0]
print(f"\nslope of ordered domination with step : {slope:+.6f} per ms")
print(f"your abstract claims this is POSITIVE ('varied proportionally to the step size').")
print(f"this prototype finds it {'POSITIVE - consistent' if slope > 1e-5 else 'NEGATIVE or FLAT - NOT consistent'}.")
print("""
If your own implementation gives a positive slope and this one does not, the difference lies in
the readout, and you should be able to say exactly which choice produces it: the recurrent gain,
the coincidence window, whether the matrix is restricted to figure channels, or normalisation.
Being able to name that is the difference between a model and a curve-fit.""")

---

## 5. Can you actually afford this experiment?

The order × asynchrony interaction is a *difference of differences*, which is the most expensive
thing in psychophysics. Before committing, find out how many listeners it needs.

The simulation below plants an effect of the size §1 predicts, runs your real analysis machinery
on synthetic listeners, and reports how often you would detect it. Move the sliders until the
power is acceptable, then read off the cost in participant-hours.

**If the honest answer is 60 listeners, that is worth knowing now.**

In [ ]:
#@title Power for the order x asynchrony interaction   { run: "auto" }
PEAK_ORDER_BENEFIT_DPRIME = 0.6  #@param {type:"slider", min:0.1, max:1.5, step:0.1}
TRIALS_PER_CELL           = 20   #@param {type:"slider", min:8, max:40, step:4}
N_LISTENERS               = 16   #@param {type:"slider", min:4, max:48, step:4}
BETWEEN_LISTENER_SD       = 0.4  #@param {type:"slider", min:0.0, max:1.0, step:0.1}
N_SIMULATIONS             = 300  #@param {type:"slider", min:100, max:600, step:100}

from scipy import stats
STEPS_P = np.array([0, 5, 10, 15, 20, 28], float)
TAU_C, TAU_S, A_BASE = 12.0, 20.0, 2.2

def pc_from_d(dp): return stats.norm.cdf(dp / np.sqrt(2))

def simulate(rng):
    """One experiment: N listeners x 6 steps x {ordered, shuffled}."""
    rows = []
    for L in range(N_LISTENERS):
        gain = max(0.1, 1 + rng.normal(0, BETWEEN_LISTENER_SD))
        for s in STEPS_P:
            coh  = np.exp(-s / TAU_C)
            stdp = (s / TAU_S) * np.exp(1 - s / TAU_S)
            for cond, dp in (("ordered",  gain * (A_BASE*coh + PEAK_ORDER_BENEFIT_DPRIME*stdp)),
                             ("shuffled", gain * (A_BASE*coh))):
                k = rng.binomial(TRIALS_PER_CELL, np.clip(pc_from_d(dp), 0.01, 0.99))
                rows.append((L, s, cond, k / TRIALS_PER_CELL))
    return rows

def interaction_p(rows):
    """Within-listener contrast: does the order benefit depend on asynchrony?"""
    ben = {}
    for L, s, cond, pc in rows:
        ben.setdefault((L, s), {})[cond] = pc
    per_listener = []
    for L in range(N_LISTENERS):
        b = np.array([ben[(L, s)]["ordered"] - ben[(L, s)]["shuffled"] for s in STEPS_P])
        w = STEPS_P - STEPS_P.mean()
        per_listener.append(float(np.dot(b, w) / np.dot(w, w)))   # slope of benefit on step
    t, p = stats.ttest_1samp(per_listener, 0.0)
    return p, float(np.mean(per_listener))

rng = np.random.default_rng(0)
ps, slopes = [], []
for _ in range(N_SIMULATIONS):
    r = simulate(rng); p, sl = interaction_p(r); ps.append(p); slopes.append(sl)
power = float(np.mean(np.array(ps) < 0.05))

null_rng = np.random.default_rng(1)
saved = PEAK_ORDER_BENEFIT_DPRIME
PEAK_ORDER_BENEFIT_DPRIME = 0.0
ps0 = [interaction_p(simulate(null_rng))[0] for _ in range(N_SIMULATIONS)]
PEAK_ORDER_BENEFIT_DPRIME = saved
fpr = float(np.mean(np.array(ps0) < 0.05))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(ps, bins=30, color="#2ca02c", alpha=0.8, label="effect present")
ax[0].hist(ps0, bins=30, color="0.6", alpha=0.6, label="no effect (null)")
ax[0].axvline(0.05, color="#d62728", lw=2, label="alpha = 0.05")
ax[0].set(xlabel="p value for the interaction", ylabel="count", title="p-value distributions")
ax[0].legend(fontsize=8)

grid_n = [4, 8, 12, 16, 24, 32, 48]
pw = []
for n in grid_n:
    saved_n = N_LISTENERS; N_LISTENERS = n
    rr = np.random.default_rng(3)
    pw.append(np.mean([interaction_p(simulate(rr))[0] < 0.05 for _ in range(120)]))
    N_LISTENERS = saved_n
ax[1].plot(grid_n, pw, "o-", lw=2)
ax[1].axhline(0.8, color="#d62728", ls="--", label="80% power")
ax[1].set(xlabel="number of listeners", ylabel="power", ylim=(0, 1.02),
          title=f"power vs sample size\n(benefit d'={PEAK_ORDER_BENEFIT_DPRIME}, {TRIALS_PER_CELL} trials/cell)")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

need = next((n for n, p in zip(grid_n, pw) if p >= 0.8), None)
mins = (2*6*2*TRIALS_PER_CELL*8.0)/60
print(f"power with {N_LISTENERS} listeners : {power:.2f}     false-positive rate: {fpr:.3f} (should be ~0.05)")
print(f"listeners needed for 80% power   : {need if need else '>48'}")
print(f"session length per listener      : about {mins:.0f} min of trials for both conditions x 6 steps")
print(f"                                   => {'' if not need else need} listeners x ~{max(1,round(mins/40))} session(s)")

---

## 6. What would falsify this, and when to walk away

Write these down now, while you have no data and therefore no motivated reasoning.

| observation | what it means | what to do |
|---|---|---|
| Order benefit is **flat** across asynchrony | learning helps, but not by sequence order — maybe just familiarity | pivot to "familiarity aids segregation"; weaker but real |
| Order benefit peaks at **0 ms** | impossible under your account; almost certainly a stimulus artefact | go back to the verification battery |
| Order benefit peaks at **3 ms or 200 ms** | the timescale is not STDP-like | keep the behaviour, drop the mechanistic claim |
| **No order benefit at any asynchrony** | consistent order does not aid grouping | the central claim is dead; report it |
| Benefit present but **only with cross-trial anchoring** | learning is slow and episodic, not within-trial | that is a *more* interesting paper about timescales |
| Model shows the crossover, listeners do not | the mechanism is available but unused | genuinely publishable as a negative constraint |

**The honest exit condition.** If, after the fixed stimulus and 8 listeners, the order benefit at
its predicted peak is under d′ = 0.2 with a confidence interval excluding 0.5, the effect is too
small to build a thesis on. Decide *now* that this is your stopping rule.

---

## 7. The plan

**Phase 0 — make the task possible (days).** You cannot test anything a listener cannot hear.
Adopt the measured configuration (60 ms tones, 21 tones/channel, 2.2–3.3 Hz), re-pilot yourself,
then two colleagues. *Gate: clear practice at the easiest condition.*

**Phase 1 — is there an order effect at all? (2 weeks).** One asynchrony, near the predicted peak.
Consistent order versus reshuffled, nothing else. 8 listeners, ~30 min each. This is a single
comparison with a clean control, and it decides whether the project has a subject.
*Gate: benefit d′ > 0.2 with a CI that excludes zero.*

**Phase 2 — the shape (6 weeks).** Only if Phase 1 passes. The full order × asynchrony
interaction, powered by §5. This is the poster and the paper's first figure.

**Phase 3 — the strong claim (in parallel).** Cross-trial anchoring (§3). Find the asynchrony
where a novel figure is at chance, and ask whether a *learned* one is detectable there. If yes,
learning does not modulate segregation — it enables it.

**Phase 4 — model and EEG.** The model prediction must be registered *before* Phase 2 data. EEG
only once the behaviour is solid; an EEG study anchored to a null behavioural effect is unreadable.

### What to submit to ARO now

Not the abstract you drafted — it reports results you do not have, and its 35 ms step is
arithmetically impossible at 3–5 Hz (seven components 35 ms apart span 240 ms; your minimum
inter-element interval is 200 ms).

Submit the **model** as the finding, the task as methods, behaviour as underway:

> *A recurrent excitatory–inhibitory model predicts that learned sequential order sustains
> auditory figure-ground segregation as component onsets separate*

That is honest, it is interesting on its own, and §4 gives you the figures for it.

In [ ]:
#@title A one-page summary of where things stand
print("""
WHAT IS SOLID
  - a stimulus generator with a verification battery that catches its own confounds
  - a measured diagnosis of why the pilot failed (figure on 8% of the interval, detector d'=0.62)
  - a fixed configuration measured at 1.6x the audibility, with one bounded, step-independent residual
  - the ordered/reshuffled contrast already implemented and spectrally matched

WHAT IS NOT
  - no behavioural data beyond one failed pilot
  - the design forbids the cross-trial learning the hypothesis is about
  - the order contrast is buried in control cells instead of being the main experiment
  - the model's prediction has not been checked against the behavioural prediction (section 4)

THE ONE SENTENCE VERSION
  Coherence binds what starts together; STDP can only learn what does NOT start together.
  So the benefit of consistent order must be zero at synchrony and grow with asynchrony.
  That crossover is the finding, and everything else is scaffolding for measuring it.

NEXT ACTION
  Phase 0. Re-pilot on the fixed configuration. Nothing else matters until a listener
  can hear the figure at the easiest condition.
""")